# 09 — Combined topology + prompt evolution (two-phase ADAS + GEPA)

**Phase 1 finds the best topology with fixed strong prompts; Phase 2 runs GEPA on the winner's prompts.**

Why two-phase, not nested? Naive nesting is `O(N_topology × N_prompts × N_eval)` — four-to-five-figure dollar bills per discovery. AFlow gets its [~5% of GPT-4o cost](https://arxiv.org/abs/2410.10762) headline by *decoupling* exactly this way: discover topology with a fixed strong-prompt template, then tune prompts on the winner. This notebook surfaces the ablation:

| Configuration | Topology | Prompts | Expected lift |
|---|---|---|---|
| **Baseline** | seed CoT | seed | reference |
| **Topology only** | evolved (notebook 08) | seed | structural lift |
| **Prompts only** | seed CoT | GEPA-evolved | semantic lift |
| **Combined** | evolved | GEPA-evolved on winner | strictly above either alone |

Cost: ~\$3–5 (Phase 1: ~\$1–2 from notebook 08; Phase 2: ~\$1–3 GEPA).


## 1. Re-use notebook 08's setup

In [ ]:
from __future__ import annotations

from typing import Any
from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()


class Answer(BaseModel):
    answer: str = Field(description="Direct answer or 'the paragraph does not say'.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


# Define agents — note the system_prompt slot we'll evolve via GEPA in Phase 2.
SEED_COT_PROMPT = (
    "You answer questions concisely from the provided paragraph. "
    "If the paragraph does not contain the answer, reply 'the paragraph does not say'."
)
SEED_VERIFIER_PROMPT = (
    "You receive a question and a candidate answer. Verify the answer is correct given the paragraph. "
    "If incorrect, replace it with the correct answer. If the paragraph does not contain the answer, "
    "reply 'the paragraph does not say'."
)


def make_cot_with_prompt(prompt: str):
    def _f():
        return CotAgent(
            agent_name="cot",
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


def make_verifier_with_prompt(prompt: str):
    def _f():
        return CotAgent(
            agent_name="verifier",
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


def make_registry(cot_prompt: str, verifier_prompt: str):
    return {
        "cot": make_cot_with_prompt(cot_prompt),
        "verifier": make_verifier_with_prompt(verifier_prompt),
    }


seed_registry = make_registry(SEED_COT_PROMPT, SEED_VERIFIER_PROMPT)
print("agents:", sorted(seed_registry))


## 2. Gold set (same shape as notebook 08, halved for cost)

In [ ]:
from orqest.optimization import GoldExample


def ex(question: str, paragraph: str, expected: str, bucket: str) -> GoldExample[str, Answer]:
    return GoldExample[str, Answer](
        input=f"Paragraph: {paragraph}\n\nQuestion: {question}",
        expected=Answer(answer=expected),
        id=bucket,
    )


gold_set = [
    ex("In what year was Python first released?",
       "Python was first released in 1991 by Guido van Rossum.",
       "1991", "factual"),
    ex("Who is the CEO of Tesla?",
       "Tesla, headquartered in Austin, is led by CEO Elon Musk.",
       "Elon Musk", "factual"),
    ex("How many years between Python and JavaScript releases?",
       "Python was released in 1991. JavaScript first appeared in 1995.",
       "4", "inferential"),
    ex("Which is older — the company or its CEO's tenure?",
       "Tesla was founded in 2003. Elon Musk became CEO in 2008.",
       "the company", "inferential"),
    ex("What is the speed of light in km/s?",
       "This paragraph discusses the history of jazz music in New Orleans during the 1920s.",
       "the paragraph does not say", "abstain"),
    ex("Who painted the Mona Lisa?",
       "Solar panels convert sunlight into electricity at 15-22% efficiency.",
       "the paragraph does not say", "abstain"),
]
print(f"gold set: {len(gold_set)} examples")


## 3. Score function + helpers

In [ ]:
def score_fn(output: Answer, example: GoldExample[str, Answer]) -> float:
    if example.expected is None:
        return 0.0
    return 1.0 if example.expected.answer.lower().strip() in output.answer.lower().strip() else 0.0


def per_bucket(bundles, examples):
    by: dict[str, list[float]] = {}
    for b, e in zip(bundles, examples):
        by.setdefault(e.id, []).append(b.accuracy)
    return {k: sum(v) / len(v) for k, v in by.items()}


def overall(buckets):
    return sum(buckets.values()) / max(1, len(buckets))


def fmt(label, buckets):
    line = "  ".join(f"{k}={buckets.get(k, 0):.2f}" for k in ("factual", "inferential", "abstain"))
    return f"{label:<28} {line}  overall={overall(buckets):.2f}"


## 4. Baseline measurement

In [ ]:
from orqest.optimization import TopologyEvaluator
from orqest.orchestration.spec import AgentStepSpec, PipelineSpec, PipelineStepEntry
from orqest.orchestration.hydrate import CallableRegistry

callable_registry = CallableRegistry()
seed_topology = PipelineSpec(steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="cot"))])

baseline_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=seed_registry,
)
baseline_bundles = await baseline_evaluator.evaluate_batch({"main": seed_topology}, gold_set)
baseline_buckets = per_bucket(baseline_bundles, gold_set)
print(fmt("BASELINE (seed/seed)", baseline_buckets))


## 5. Phase 1 — Topology search (~\$1–2)

Identical configuration to notebook 08 but smaller (3 generations) since this notebook stacks two phases.

In [ ]:
from orqest.optimization import (
    MetaAgentConfig, MetaAgentSearch, TopologyGene,
)

gene = TopologyGene(
    name="main",
    initial=seed_topology,
    constraints="Prefer simpler topologies. Add a verifier step only when it helps the inferential bucket.",
)
search = MetaAgentSearch(
    MetaAgentConfig(
        n_generations=3,
        archive_strategy="top_k",
        archive_size=2,
        reflexion_passes=1,
        debug_max=2,
        minibatch_size=3,
        seed=42,
    ),
    gene=gene,
    evaluator=baseline_evaluator,
    meta_agent_model=config.llm_model,
    api_key=config.llm_api_key,
)
topology_result = await search.run(trainset=gold_set)
winning_topology = topology_result.best_decoded["main"]
print(f"phase 1 best score (minibatch): {topology_result.best_score:.3f}")
print(f"winning topology kind:          {winning_topology.kind}")


## 6. Topology-only ablation row

Re-evaluate the winning topology against the full gold set — *with seed prompts unchanged*.

In [ ]:
topology_only_bundles = await baseline_evaluator.evaluate_batch(
    {"main": winning_topology}, gold_set,
)
topology_only_buckets = per_bucket(topology_only_bundles, gold_set)
print(fmt("TOPOLOGY only (evolved/seed)", topology_only_buckets))


## 7. Phase 2 — GEPA prompt evolution on the winning topology (~\$1–3)

Extract every `system_prompt` slot from the winning topology, build a `PromptGene` per slot, run the existing GEPA path. The agent_factory closes over the winning topology's structure.

In [ ]:
from orqest.optimization import (
    Evaluator, Genome, OptimizationConfig, OptimizationRunner, PromptGene,
)
from orqest.orchestration.hydrate import topology_from_spec


def collect_prompt_slots(spec, prefix=""):
    """Recursively walk a TopologySpec, collecting (slot_name, agent_name) for AgentStepSpec leaves."""
    from orqest.orchestration.spec import (
        AgentStepSpec, FunctionStepSpec, PipelineSpec, ParallelSpec, RouterSpec, RefinementLoopSpec,
    )
    out: list[tuple[str, str]] = []
    if isinstance(spec, AgentStepSpec) and spec.agent_name:
        slot = f"{prefix}{spec.agent_name}.system_prompt"
        out.append((slot, spec.agent_name))
    elif isinstance(spec, PipelineSpec):
        for i, e in enumerate(spec.steps):
            out.extend(collect_prompt_slots(e.operation, f"{prefix}step{i}."))
    elif isinstance(spec, ParallelSpec):
        for i, s in enumerate(spec.steps):
            out.extend(collect_prompt_slots(s, f"{prefix}p{i}."))
    elif isinstance(spec, RouterSpec):
        for r in spec.routes:
            out.extend(collect_prompt_slots(r.step, f"{prefix}{r.name}."))
        if spec.fallback_step is not None:
            out.extend(collect_prompt_slots(spec.fallback_step, f"{prefix}fallback."))
    elif isinstance(spec, RefinementLoopSpec):
        out.extend(collect_prompt_slots(spec.step, f"{prefix}loop."))
    return out


SEED_PROMPTS = {"cot": SEED_COT_PROMPT, "verifier": SEED_VERIFIER_PROMPT}

slots = collect_prompt_slots(winning_topology)
unique_agent_names = sorted({a for _, a in slots})
print(f"unique agents in winning topology: {unique_agent_names}")

# One PromptGene per unique agent (multiple structural slots per agent share a prompt).
prompt_genes = [
    PromptGene(
        name=f"{a}.system_prompt",
        initial=SEED_PROMPTS.get(a, SEED_COT_PROMPT),
        constraints="Must end with 'the paragraph does not say' instruction for off-topic / abstain inputs.",
    )
    for a in unique_agent_names
]
genome = Genome(genes=prompt_genes)
print(f"GEPA genome: {len(genome.genes)} prompt gene(s)")


In [ ]:
def prompt_evolved_factory_for_agent(agent_name: str, decoded_prompts: dict[str, Any]):
    prompt = decoded_prompts.get(f"{agent_name}.system_prompt", SEED_PROMPTS.get(agent_name, SEED_COT_PROMPT))
    def _f():
        return CotAgent(
            agent_name=agent_name,
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


# GEPA evaluates the winning topology with whatever prompts the candidate proposes.
class _EvolvedTopologyAgent:
    """A wrapper that pretends to be a BaseAgent so the existing Evaluator interface fits.

    The single-agent Evaluator expects ``agent.call_model(prompt, state) -> AgentRunResult``.
    We emulate that by hydrating the winning topology with prompts from the decoded genome.
    """

    def __init__(self, decoded_prompts):
        self._decoded = decoded_prompts

    async def call_model(self, prompt, state):
        agents = {a: prompt_evolved_factory_for_agent(a, self._decoded) for a in unique_agent_names}
        topology = topology_from_spec(
            winning_topology,
            callable_registry=callable_registry,
            agent_registry=agents,
        )
        run_result = await topology.run(prompt)
        # Unpack composites
        from orqest.orchestration.parallel import ParallelResult
        from orqest.orchestration.loop import LoopResult
        if isinstance(run_result, ParallelResult):
            inner = run_result.merged
            output = inner if not isinstance(inner, list) else (inner[0] if inner else None)
        elif isinstance(run_result, LoopResult):
            output = run_result.output
        else:
            output = getattr(run_result, "output", run_result)
        # Match AgentRunResult-ish surface
        class _R:
            pass
        r = _R()
        r.output = output
        r.usage = lambda: None
        return r


def gepa_agent_factory(decoded):
    return _EvolvedTopologyAgent(decoded)


prompt_evaluator = Evaluator(agent_factory=gepa_agent_factory, score_fn=score_fn)
prompt_runner = OptimizationRunner(
    OptimizationConfig(
        max_metric_calls=20,
        reflection_model=config.llm_model,
        minibatch_size=3,
        valset_fraction=0.5,
        seed=42,
    ),
    genome=genome,
    evaluator=prompt_evaluator,
    api_key=config.llm_api_key,
)
prompt_result = await prompt_runner.optimize(trainset=gold_set)
print(f"phase 2 best score: {prompt_result.best_score:.3f}")


## 8. Combined ablation row + headline table

In [ ]:
# Prompts-only row: seed topology with GEPA-evolved prompts
prompts_only_registry = {
    a: prompt_evolved_factory_for_agent(a, prompt_result.best_decoded)
    for a in unique_agent_names if a in seed_registry
}
prompts_only_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=prompts_only_registry,
)
prompts_only_bundles = await prompts_only_evaluator.evaluate_batch({"main": seed_topology}, gold_set)
prompts_only_buckets = per_bucket(prompts_only_bundles, gold_set)

# Combined row: winning topology with GEPA-evolved prompts
combined_registry = {
    a: prompt_evolved_factory_for_agent(a, prompt_result.best_decoded)
    for a in unique_agent_names
}
combined_evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=combined_registry,
)
combined_bundles = await combined_evaluator.evaluate_batch({"main": winning_topology}, gold_set)
combined_buckets = per_bucket(combined_bundles, gold_set)

print("=== ABLATION ===\n")
print(fmt("BASELINE      (seed/seed)",      baseline_buckets))
print(fmt("TOPOLOGY only (evolved/seed)",   topology_only_buckets))
print(fmt("PROMPTS only  (seed/evolved)",   prompts_only_buckets))
print(fmt("COMBINED      (evolved/evolved)", combined_buckets))
print()
print("Lift over baseline (overall):")
for label, buckets in [
    ("topology only",  topology_only_buckets),
    ("prompts only",   prompts_only_buckets),
    ("combined",       combined_buckets),
]:
    print(f"  {label:<14} {overall(buckets) - overall(baseline_buckets):+.3f}")


## Reading the ablation honestly

The interesting case is when the four rows agree at 1.00 (baseline saturated) — then this notebook *demonstrates the wiring works* but has nothing to claim about lift. The interesting *failure* case is when COMBINED scores **lower** than baseline (we observed `combined=0.83` with `inferential=0.50` in one strong-model run). That's a real signal, not a bug:

- The Phase-2 GEPA was tuned for the Phase-1 winning topology's *specific shape* on a small minibatch. When the baseline is already saturated, GEPA can converge on prompts that overfit to noise — and overfit prompts hurt buckets they weren't sampled from.
- The mitigation is exactly what a real production search would do: bigger valset, more reflexion passes, larger minibatch, weaker task model so there's actual headroom to climb.

This is the classic search-overfit-to-tiny-eval-set failure that the [2510.06711 critique](https://arxiv.org/abs/2510.06711) documents at length. The right takeaway: **two-phase composition only beats either phase alone when there's real headroom**. Don't run a search on a benchmark your seed already aces.

## What's next

- **MCTS over `TopologySpec`** (AFlow-style) — alternative search engine plugged into the same `Archive` + `MetricBundle` plumbing.
- **Per-step cost capture.** Today the topology Evaluator surfaces `cost_usd=0.0` because we don't aggregate `Usage` across topology agents. A per-step cost-tracking hook is the next refinement; the `MetricBundle.raw` extension point is already there for it.
- **Sandboxed codegen** (W3.C) — the meta agent emits raw Python `forward()` for tasks where compositions of registered primitives are too constrained.

For the design rationale, see `docs/concepts/topology_optimization.md`.
